# Matched Control Candidates

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
#!/usr/bin/env python3
"""Build matched-control candidate areas and Vivacity countline shortlists.

This does not claim that controls are final. It creates a transparent shortlist:
candidate LSOA 2021 areas are ranked by contextual similarity to each treated
sensor LSOA, and available Vivacity countlines in those candidate LSOAs are
listed for possible future data download.
"""

from __future__ import annotations

import csv
import json
import math
import re
from collections import defaultdict
from pathlib import Path


BASE = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CONTEXT = BASE / "context_data"
PROCESSED = CONTEXT / "processed"
RAW_META = CONTEXT / "raw" / "vivacity_metadata"
MATCH_DIR = PROCESSED / "matching"
MAP_DIR = CONTEXT / "maps"
QA_DIR = CONTEXT / "quality_checks"

LSOA_BOUNDARIES = PROCESSED / "lsoa" / "lcr_lsoa_2021_boundaries.geojson"
LSOA_CONTEXT = PROCESSED / "lsoa" / "lsoa2021_lcr_context_wide.csv"
SENSOR_CONTEXT = PROCESSED / "sensors" / "vivacity_countline_lsoa2021_context_joined.csv"
COUNTLINES_JSON = RAW_META / "countlines.json"
HARDWARE_TXT = RAW_META / "hardware_positions_with_status.txt"

TARGET_COUNTLINE_IDS = {
    46676, 46686, 46687, 46688,
    16373, 16374, 16393, 16394, 23779,
    51794, 51795, 51796, 51817, 51818,
    23799, 23800, 23801,
    47768, 47769, 47772, 47773,
}

MATCH_VARIABLES = [
    ("imd_decile", 2.0),
    ("income_decile", 1.4),
    ("employment_decile", 1.0),
    ("census_population_density_ppsqkm", 1.0),
    ("census_pct_households_no_car", 1.2),
    ("census_pct_commute_bicycle", 0.8),
    ("census_pct_commute_on_foot", 0.8),
]


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        for row in rows:
            for key in row:
                if key not in fieldnames:
                    fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def to_float(value: object) -> float | None:
    if value in (None, "", "NA", "NaN"):
        return None
    try:
        number = float(value)
        if math.isnan(number) or math.isinf(number):
            return None
        return number
    except (TypeError, ValueError):
        return None


def midpoint(coords: list[list[float]]) -> tuple[float | None, float | None]:
    if not coords:
        return None, None
    return sum(x[0] for x in coords) / len(coords), sum(x[1] for x in coords) / len(coords)


def haversine_km(lon1: float, lat1: float, lon2: float, lat2: float) -> float:
    radius = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return radius * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def point_in_ring(lon: float, lat: float, ring: list[list[float]]) -> bool:
    inside = False
    j = len(ring) - 1
    for i in range(len(ring)):
        xi, yi = ring[i][0], ring[i][1]
        xj, yj = ring[j][0], ring[j][1]
        if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / ((yj - yi) or 1e-15) + xi):
            inside = not inside
        j = i
    return inside


def point_in_polygon(lon: float, lat: float, coordinates: list) -> bool:
    if not coordinates or not point_in_ring(lon, lat, coordinates[0]):
        return False
    return not any(point_in_ring(lon, lat, hole) for hole in coordinates[1:])


def point_in_geometry(lon: float, lat: float, geometry: dict) -> bool:
    if geometry.get("type") == "Polygon":
        return point_in_polygon(lon, lat, geometry.get("coordinates") or [])
    if geometry.get("type") == "MultiPolygon":
        return any(point_in_polygon(lon, lat, poly) for poly in geometry.get("coordinates") or [])
    return False


def parse_hardware() -> dict[int, dict]:
    if not HARDWARE_TXT.exists():
        return {}
    text = HARDWARE_TXT.read_text(encoding="utf-8")
    payload = json.loads(text[text.find("{"):])
    by_viewpoint = {}
    for hardware in payload.get("data", []):
        for viewpoint_id in (hardware.get("child_entities") or {}).get("viewpoint_ids") or []:
            by_viewpoint[int(viewpoint_id)] = hardware
    return by_viewpoint


def classify_route(name: str) -> str:
    lower = (name or "").lower()
    if "path" in lower or "cycle" in lower:
        return "Path/cycle facility"
    if "road" in lower:
        return "Road"
    return "Other"


def load_lsoa_context() -> tuple[dict[str, dict], dict[str, dict]]:
    geo = json.loads(LSOA_BOUNDARIES.read_text(encoding="utf-8"))
    geometry_by_code = {}
    props_by_code = {}
    for feature in geo["features"]:
        props = feature.get("properties") or {}
        code = props.get("LSOA21CD")
        if not code:
            continue
        geometry_by_code[code] = feature["geometry"]
        props_by_code[code] = props

    context_by_code = {}
    for row in read_csv(LSOA_CONTEXT):
        code = row.get("analysis_lsoa21cd")
        if not code or code not in props_by_code:
            continue
        props = props_by_code[code]
        merged = dict(row)
        merged.update(
            {
                "analysis_lsoa21cd": code,
                "analysis_lsoa21nm": row.get("analysis_lsoa21nm") or props.get("LSOA21NM"),
                "lsoa_centroid_lat": props.get("LAT"),
                "lsoa_centroid_lon": props.get("LONG"),
                "lsoa21_urban_rural": props.get("Urban_rura", ""),
            }
        )
        context_by_code[code] = merged
    return context_by_code, geometry_by_code


def treated_lsoa_profiles(context_by_code: dict[str, dict]) -> list[dict[str, object]]:
    sensor_rows = read_csv(SENSOR_CONTEXT)
    by_lsoa: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in sensor_rows:
        by_lsoa[row["analysis_lsoa21cd"]].append(row)

    rows = []
    for code, group in sorted(by_lsoa.items()):
        context = context_by_code[code]
        schemes = sorted({row["scheme_id"] for row in group})
        rows.append(
            {
                "treated_lsoa21cd": code,
                "treated_lsoa21nm": context.get("analysis_lsoa21nm"),
                "treated_lad": context.get("LAD22NM"),
                "schemes": "; ".join(schemes),
                "road_groups": "; ".join(sorted({row["road_group"] for row in group})),
                "countlines": len(group),
                "route_types": "; ".join(sorted({row["route_type"] for row in group})),
                **{name: context.get(name, "") for name, _ in MATCH_VARIABLES},
                "lsoa_centroid_lat": context.get("lsoa_centroid_lat"),
                "lsoa_centroid_lon": context.get("lsoa_centroid_lon"),
            }
        )
    return rows


def standardisation(context_by_code: dict[str, dict]) -> dict[str, tuple[float, float]]:
    stats = {}
    for name, _ in MATCH_VARIABLES:
        values = [to_float(row.get(name)) for row in context_by_code.values()]
        values = [x for x in values if x is not None]
        mean = sum(values) / len(values)
        var = sum((x - mean) ** 2 for x in values) / max(len(values) - 1, 1)
        sd = math.sqrt(var) or 1.0
        stats[name] = (mean, sd)
    return stats


def build_matches(context_by_code: dict[str, dict], treated_rows: list[dict[str, object]]) -> list[dict[str, object]]:
    treated_codes = {row["treated_lsoa21cd"] for row in treated_rows}
    stats = standardisation(context_by_code)
    candidates = [row for code, row in context_by_code.items() if code not in treated_codes]
    match_rows = []

    for treated in treated_rows:
        tcode = treated["treated_lsoa21cd"]
        tcontext = context_by_code[tcode]
        tlon = to_float(tcontext.get("lsoa_centroid_lon"))
        tlat = to_float(tcontext.get("lsoa_centroid_lat"))
        scored = []
        for candidate in candidates:
            ccode = candidate["analysis_lsoa21cd"]
            clon = to_float(candidate.get("lsoa_centroid_lon"))
            clat = to_float(candidate.get("lsoa_centroid_lat"))
            weighted_sq = 0.0
            weight_sum = 0.0
            missing = []
            for name, weight in MATCH_VARIABLES:
                tv = to_float(tcontext.get(name))
                cv = to_float(candidate.get(name))
                if tv is None or cv is None:
                    missing.append(name)
                    continue
                mean, sd = stats[name]
                weighted_sq += weight * (((tv - mean) / sd) - ((cv - mean) / sd)) ** 2
                weight_sum += weight
            if weight_sum == 0:
                continue
            attribute_distance = math.sqrt(weighted_sq / weight_sum)
            geo_distance = (
                haversine_km(tlon, tlat, clon, clat)
                if None not in (tlon, tlat, clon, clat)
                else None
            )
            same_lad = tcontext.get("LAD22NM") == candidate.get("LAD22NM")
            imd_diff = abs((to_float(tcontext.get("imd_decile")) or 0) - (to_float(candidate.get("imd_decile")) or 0))
            score = attribute_distance + (0.015 * (geo_distance or 0)) - (0.15 if same_lad else 0)
            scored.append(
                {
                    "treated_lsoa21cd": tcode,
                    "treated_lsoa21nm": tcontext.get("analysis_lsoa21nm"),
                    "treated_lad": tcontext.get("LAD22NM"),
                    "treated_schemes": treated.get("schemes"),
                    "treated_road_groups": treated.get("road_groups"),
                    "candidate_lsoa21cd": ccode,
                    "candidate_lsoa21nm": candidate.get("analysis_lsoa21nm"),
                    "candidate_lad": candidate.get("LAD22NM"),
                    "same_lad": same_lad,
                    "attribute_distance": attribute_distance,
                    "geo_distance_km": geo_distance,
                    "match_score": score,
                    "imd_decile_diff": imd_diff,
                    "missing_match_variables": "; ".join(missing),
                    **{f"treated_{name}": tcontext.get(name, "") for name, _ in MATCH_VARIABLES},
                    **{f"candidate_{name}": candidate.get(name, "") for name, _ in MATCH_VARIABLES},
                    "candidate_lsoa_centroid_lat": candidate.get("lsoa_centroid_lat"),
                    "candidate_lsoa_centroid_lon": candidate.get("lsoa_centroid_lon"),
                }
            )
        scored.sort(key=lambda x: (x["match_score"], x["attribute_distance"]))
        for rank, row in enumerate(scored[:50], start=1):
            row["candidate_rank"] = rank
            row["suggested_priority"] = "High" if rank <= 5 else "Medium" if rank <= 15 else "Longlist"
            match_rows.append(row)
    return match_rows


def build_all_countlines(context_by_code: dict[str, dict], geometry_by_code: dict[str, dict]) -> list[dict[str, object]]:
    countlines = json.loads(COUNTLINES_JSON.read_text(encoding="utf-8")).get("data", [])
    hardware_by_viewpoint = parse_hardware()
    rows = []
    for countline in countlines:
        coords = ((countline.get("geometry") or {}).get("gps") or {}).get("coordinates") or []
        lon, lat = midpoint(coords)
        if lon is None or lat is None:
            continue
        matched_code = ""
        for code, geometry in geometry_by_code.items():
            if point_in_geometry(lon, lat, geometry):
                matched_code = code
                break
        if not matched_code:
            continue
        context = context_by_code.get(matched_code, {})
        viewpoint_id = countline.get("viewpoint_id")
        hardware = hardware_by_viewpoint.get(int(viewpoint_id)) if viewpoint_id is not None else {}
        rows.append(
            {
                "countline_id": countline.get("id"),
                "countline_name": countline.get("name"),
                "viewpoint_id": viewpoint_id,
                "route_type_guess": classify_route(countline.get("name", "")),
                "countline_midpoint_lon": lon,
                "countline_midpoint_lat": lat,
                "analysis_lsoa21cd": matched_code,
                "analysis_lsoa21nm": context.get("analysis_lsoa21nm"),
                "LAD22NM": context.get("LAD22NM"),
                "imd_decile": context.get("imd_decile"),
                "income_decile": context.get("income_decile"),
                "census_population_density_ppsqkm": context.get("census_population_density_ppsqkm"),
                "census_pct_households_no_car": context.get("census_pct_households_no_car"),
                "hardware_id": hardware.get("id", ""),
                "hardware_sensor_number": hardware.get("sensor_number", ""),
                "hardware_name": hardware.get("name", ""),
                "hardware_status": hardware.get("status", ""),
                "is_dissertation_intervention_countline": int(countline.get("id")) in TARGET_COUNTLINE_IDS,
            }
        )
    return rows


def build_candidate_countlines(all_countlines: list[dict[str, object]], matches: list[dict[str, object]]) -> list[dict[str, object]]:
    top_matches = [row for row in matches if int(row["candidate_rank"]) <= 25]
    match_by_lsoa: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in top_matches:
        match_by_lsoa[row["candidate_lsoa21cd"]].append(row)

    out = []
    for countline in all_countlines:
        if countline.get("is_dissertation_intervention_countline"):
            continue
        code = countline.get("analysis_lsoa21cd")
        for match in match_by_lsoa.get(code, []):
            row = dict(countline)
            row.update(
                {
                    "matched_to_treated_lsoa21cd": match["treated_lsoa21cd"],
                    "matched_to_treated_lsoa21nm": match["treated_lsoa21nm"],
                    "matched_to_schemes": match["treated_schemes"],
                    "matched_to_road_groups": match["treated_road_groups"],
                    "candidate_rank": match["candidate_rank"],
                    "suggested_priority": match["suggested_priority"],
                    "match_score": match["match_score"],
                    "attribute_distance": match["attribute_distance"],
                    "geo_distance_km": match["geo_distance_km"],
                    "same_lad": match["same_lad"],
                    "download_note": "Potential control countline. Check dashboard history and intervention status before downloading.",
                }
            )
            out.append(row)
    out.sort(key=lambda x: (x["matched_to_treated_lsoa21cd"], int(x["candidate_rank"]), str(x["countline_name"])))
    return out


def build_download_shortlist(candidate_countlines: list[dict[str, object]], per_scheme: int = 15) -> list[dict[str, object]]:
    grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    seen: set[tuple[str, str]] = set()
    for row in candidate_countlines:
        scheme = str(row.get("matched_to_schemes", ""))
        key = (scheme, str(row.get("countline_id", "")))
        if key in seen:
            continue
        seen.add(key)
        grouped[scheme].append(row)

    shortlist = []
    for scheme, rows in sorted(grouped.items()):
        rows.sort(key=lambda x: (int(x["candidate_rank"]), float(x["match_score"]), str(x["countline_name"])))
        for rank, row in enumerate(rows[:per_scheme], start=1):
            out = {
                "download_shortlist_rank_within_scheme": rank,
                "matched_to_schemes": row.get("matched_to_schemes"),
                "matched_to_treated_lsoa21nm": row.get("matched_to_treated_lsoa21nm"),
                "matched_to_road_groups": row.get("matched_to_road_groups"),
                "candidate_lsoa_rank": row.get("candidate_rank"),
                "suggested_priority": row.get("suggested_priority"),
                "candidate_lsoa21cd": row.get("analysis_lsoa21cd"),
                "candidate_lsoa21nm": row.get("analysis_lsoa21nm"),
                "candidate_lad": row.get("LAD22NM"),
                "countline_id": row.get("countline_id"),
                "countline_name": row.get("countline_name"),
                "route_type_guess": row.get("route_type_guess"),
                "hardware_name": row.get("hardware_name"),
                "hardware_status": row.get("hardware_status"),
                "match_score": row.get("match_score"),
                "same_lad": row.get("same_lad"),
                "geo_distance_km": row.get("geo_distance_km"),
                "download_note": "Download the same date range as the treated scheme if dashboard history exists; manually check this countline is not itself an intervention site.",
            }
            shortlist.append(out)
    return shortlist


def make_map_geojson(context_by_code: dict[str, dict], geometry_by_code: dict[str, dict], treated: list[dict], matches: list[dict], countlines: list[dict]) -> tuple[dict, dict, dict]:
    treated_codes = {row["treated_lsoa21cd"] for row in treated}
    top_match_rows = [row for row in matches if int(row["candidate_rank"]) <= 5]
    top_codes = {row["candidate_lsoa21cd"] for row in top_match_rows}
    top_by_code = defaultdict(list)
    for row in top_match_rows:
        top_by_code[row["candidate_lsoa21cd"]].append(row)

    treated_features = []
    for code in treated_codes:
        context = context_by_code[code]
        treated_features.append({
            "type": "Feature",
            "geometry": geometry_by_code[code],
            "properties": {
                "type": "treated",
                "lsoa21cd": code,
                "lsoa21nm": context.get("analysis_lsoa21nm"),
                "lad": context.get("LAD22NM"),
                "imd_decile": context.get("imd_decile"),
            },
        })

    control_features = []
    for code in top_codes:
        context = context_by_code[code]
        links = top_by_code[code]
        control_features.append({
            "type": "Feature",
            "geometry": geometry_by_code[code],
            "properties": {
                "type": "matched_control_candidate",
                "lsoa21cd": code,
                "lsoa21nm": context.get("analysis_lsoa21nm"),
                "lad": context.get("LAD22NM"),
                "imd_decile": context.get("imd_decile"),
                "matched_to": "; ".join(sorted({x["treated_lsoa21nm"] for x in links})),
                "best_rank": min(int(x["candidate_rank"]) for x in links),
                "best_score": min(float(x["match_score"]) for x in links),
            },
        })

    candidate_codes = top_codes
    point_features = []
    for row in countlines:
        if row.get("analysis_lsoa21cd") not in candidate_codes:
            continue
        point_features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [to_float(row["countline_midpoint_lon"]), to_float(row["countline_midpoint_lat"])]},
            "properties": {
                "countline_id": row.get("countline_id"),
                "countline_name": row.get("countline_name"),
                "route_type_guess": row.get("route_type_guess"),
                "lsoa21cd": row.get("analysis_lsoa21cd"),
                "lsoa21nm": row.get("analysis_lsoa21nm"),
                "lad": row.get("LAD22NM"),
            },
        })

    return (
        {"type": "FeatureCollection", "features": treated_features},
        {"type": "FeatureCollection", "features": control_features},
        {"type": "FeatureCollection", "features": point_features},
    )


def build_html(treated_geo: dict, control_geo: dict, point_geo: dict) -> str:
    return f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Matched Control Candidate LSOAs</title>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <style>
    html, body, #map {{ height: 100%; margin: 0; font-family: Arial, Helvetica, sans-serif; }}
    .panel, .legend {{
      background: rgba(255,255,255,0.94);
      border: 1px solid #D1D5DB;
      border-radius: 6px;
      box-shadow: 0 2px 10px rgba(0,0,0,0.12);
      color: #111827;
    }}
    .panel {{ position:absolute; top:12px; left:52px; z-index:800; max-width:430px; padding:10px 12px; }}
    .panel h1 {{ font-size:16px; margin:0 0 4px; }}
    .panel p {{ font-size:12px; margin:0; line-height:1.35; color:#374151; }}
    .legend {{ padding:10px 12px; font-size:12px; line-height:1.35; }}
    .row {{ display:flex; gap:6px; align-items:center; margin:3px 0; }}
    .swatch {{ width:14px; height:14px; border:1px solid #4B5563; }}
    table {{ border-collapse:collapse; min-width:250px; }}
    th {{ text-align:left; padding:2px 8px 2px 0; color:#374151; white-space:nowrap; }}
    td {{ padding:2px 0; }}
  </style>
</head>
<body>
<div id="map"></div>
<div class="panel">
  <h1>Matched Control Candidate Areas</h1>
  <p>Top matched LSOA 2021 candidates for the treated sensor LSOAs. These are area/context matches only; dashboard history and scheme status still need manual checking before using them as controls.</p>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const treated = {json.dumps(treated_geo, separators=(",", ":"))};
const controls = {json.dumps(control_geo, separators=(",", ":"))};
const points = {json.dumps(point_geo, separators=(",", ":"))};
const map = L.map('map', {{ preferCanvas: true }});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
  maxZoom: 19,
  attribution: '&copy; OpenStreetMap contributors'
}}).addTo(map);
function esc(v) {{ return (v ?? '').toString().replace(/[&<>"']/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[ch])); }}
function popup(rows) {{ return '<table>' + rows.map(r => `<tr><th>${{esc(r[0])}}</th><td>${{esc(r[1])}}</td></tr>`).join('') + '</table>'; }}
const treatedLayer = L.geoJSON(treated, {{
  style: {{ color:'#111827', weight:2.5, fillColor:'#F97316', fillOpacity:0.42 }},
  onEachFeature: (f,l) => l.bindPopup(popup([
    ['Treated LSOA', `${{f.properties.lsoa21nm}} (${{f.properties.lsoa21cd}})`],
    ['LAD', f.properties.lad],
    ['IMD decile', f.properties.imd_decile]
  ]))
}}).addTo(map);
const controlLayer = L.geoJSON(controls, {{
  style: {{ color:'#1D4ED8', weight:1.8, fillColor:'#60A5FA', fillOpacity:0.32 }},
  onEachFeature: (f,l) => l.bindPopup(popup([
    ['Candidate LSOA', `${{f.properties.lsoa21nm}} (${{f.properties.lsoa21cd}})`],
    ['Matched to', f.properties.matched_to],
    ['Best rank', f.properties.best_rank],
    ['Best score', Number(f.properties.best_score).toFixed(3)],
    ['LAD', f.properties.lad],
    ['IMD decile', f.properties.imd_decile]
  ]))
}}).addTo(map);
const pointLayer = L.geoJSON(points, {{
  pointToLayer: (f, latlng) => L.circleMarker(latlng, {{ radius:4, color:'#111827', weight:1, fillColor:'#10B981', fillOpacity:0.8 }}),
  onEachFeature: (f,l) => l.bindPopup(popup([
    ['Countline', `${{f.properties.countline_name}} (${{f.properties.countline_id}})`],
    ['Route guess', f.properties.route_type_guess],
    ['LSOA', `${{f.properties.lsoa21nm}} (${{f.properties.lsoa21cd}})`],
    ['LAD', f.properties.lad]
  ]))
}}).addTo(map);
L.control.layers({{'OpenStreetMap': Object.values(map._layers)[0]}}, {{
  'Treated sensor LSOAs': treatedLayer,
  'Top matched control LSOAs': controlLayer,
  'Vivacity countlines in top control LSOAs': pointLayer
}}, {{ collapsed:false }}).addTo(map);
const bounds = treatedLayer.getBounds().extend(controlLayer.getBounds());
if (bounds.isValid()) map.fitBounds(bounds.pad(0.2));
const legend = L.control({{position:'bottomright'}});
legend.onAdd = () => {{
  const div = L.DomUtil.create('div', 'legend');
  div.innerHTML = '<strong>Legend</strong><div class="row"><span class="swatch" style="background:#F97316"></span>Treated sensor LSOA</div><div class="row"><span class="swatch" style="background:#60A5FA"></span>Matched control LSOA candidate</div><div class="row"><span class="swatch" style="background:#10B981;border-radius:50%"></span>Vivacity countline candidate</div>';
  return div;
}};
legend.addTo(map);
</script>
</body>
</html>"""


def main() -> None:
    MATCH_DIR.mkdir(parents=True, exist_ok=True)
    MAP_DIR.mkdir(parents=True, exist_ok=True)
    QA_DIR.mkdir(parents=True, exist_ok=True)

    context_by_code, geometry_by_code = load_lsoa_context()
    treated = treated_lsoa_profiles(context_by_code)
    matches = build_matches(context_by_code, treated)
    all_countlines = build_all_countlines(context_by_code, geometry_by_code)
    candidate_countlines = build_candidate_countlines(all_countlines, matches)
    download_shortlist = build_download_shortlist(candidate_countlines)

    write_csv(MATCH_DIR / "treated_sensor_lsoa_profiles.csv", treated)
    write_csv(MATCH_DIR / "lsoa2021_matched_control_candidates.csv", matches)
    write_csv(MATCH_DIR / "all_vivacity_countlines_lsoa2021_context.csv", all_countlines)
    write_csv(MATCH_DIR / "vivacity_control_countline_candidates.csv", candidate_countlines)
    write_csv(MATCH_DIR / "vivacity_control_download_shortlist.csv", download_shortlist)

    treated_geo, control_geo, point_geo = make_map_geojson(context_by_code, geometry_by_code, treated, matches, all_countlines)
    (MAP_DIR / "matched_control_candidate_map.html").write_text(build_html(treated_geo, control_geo, point_geo), encoding="utf-8")
    geojson_dir = MAP_DIR / "geojson"
    geojson_dir.mkdir(parents=True, exist_ok=True)
    (geojson_dir / "treated_sensor_lsoas.geojson").write_text(json.dumps(treated_geo, indent=2), encoding="utf-8")
    (geojson_dir / "top_matched_control_lsoas.geojson").write_text(json.dumps(control_geo, indent=2), encoding="utf-8")
    (geojson_dir / "top_control_candidate_countlines.geojson").write_text(json.dumps(point_geo, indent=2), encoding="utf-8")

    qa_rows = [
        {"check": "treated_lsoas", "result": len(treated), "passed": len(treated) == 6, "notes": "Unique LSOA 2021 areas containing dissertation countlines."},
        {"check": "matched_lsoa_rows", "result": len(matches), "passed": len(matches) == len(treated) * 50, "notes": "Top 50 matched LSOA candidates per treated LSOA."},
        {"check": "all_vivacity_countlines_in_lcr_lsoas", "result": len(all_countlines), "passed": len(all_countlines) > 0, "notes": "All available Vivacity countline metadata spatially joined to LCR LSOA 2021."},
        {"check": "control_candidate_countline_rows", "result": len(candidate_countlines), "passed": len(candidate_countlines) > 0, "notes": "Non-intervention countlines in top-25 matched candidate LSOAs."},
        {"check": "control_download_shortlist_rows", "result": len(download_shortlist), "passed": len(download_shortlist) > 0, "notes": "Best available candidate countlines per treated scheme for manual dashboard checking/download."},
        {"check": "map_created", "result": str(MAP_DIR / "matched_control_candidate_map.html"), "passed": (MAP_DIR / "matched_control_candidate_map.html").exists(), "notes": "OSM/Leaflet map of treated and top matched control candidate areas."},
    ]
    write_csv(QA_DIR / "matched_control_candidates_qa.csv", qa_rows, ["check", "result", "passed", "notes"])
    summary = {
        "treated_lsoas": len(treated),
        "matched_lsoa_rows": len(matches),
        "all_vivacity_countlines_in_lcr_lsoas": len(all_countlines),
        "control_candidate_countline_rows": len(candidate_countlines),
        "control_download_shortlist_rows": len(download_shortlist),
        "outputs": str(MATCH_DIR),
        "map": str(MAP_DIR / "matched_control_candidate_map.html"),
    }
    (QA_DIR / "matched_control_candidates_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
